In [1]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

In [2]:
DATAFILE = "/mnt/data_storage/datasets/morphology/morphodict-k-forms.csv"

In [3]:
df = pd.read_csv(DATAFILE, sep="\t", index_col=None, names=["form", "segments", "pos", "lemma"])

In [4]:
df.head()

,form,segments,pos,lemma
0,прикопанную,при:PREF/коп:ROOT/а:SUFF/нн:SUFF/ую:END,INFN,прикопать
1,покрытосемянно,по:PREF/кры:PREF/т:SUFF/о:LINK/сем:ROOT/ян:SUF...,ADJF,покрытосемянный
2,переплавившемся,пере:PREF/плав:ROOT/и:SUFF/вш:SUFF/ем:END/ся:POST,INFN,переплавиться
3,вспомянете,вс:PREF/по:PREF/мя:ROOT/н:SUFF/ете:END,INFN,вспомянуть
4,оближет,об:PREF/лиж:ROOT/ет:END,INFN,облизать


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1392790 entries, 0 to 1392789
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype
---  ------    --------------    -----
 0   form      1392790 non-null  str  
 1   segments  1392790 non-null  str  
 2   pos       1392790 non-null  str  
 3   lemma     1392790 non-null  str  
dtypes: str(4)
memory usage: 42.5 MB


In [6]:
segments = [s.split("/") for s in df["segments"].tolist()]
segments = [[s.split(":")[0] for s in arr] for arr in segments]

In [7]:
segments[:5]

[['при', 'коп', 'а', 'нн', 'ую'],
 ['по', 'кры', 'т', 'о', 'сем', 'ян', 'н', 'о'],
 ['пере', 'плав', 'и', 'вш', 'ем', 'ся'],
 ['вс', 'по', 'мя', 'н', 'ете'],
 ['об', 'лиж', 'ет']]

In [8]:
texts = df["form"].tolist()

In [9]:
assert len(segments) == len(texts)

In [10]:
texts_segments = list(zip(texts, segments))

In [11]:
texts_segments[:5]

[('прикопанную', ['при', 'коп', 'а', 'нн', 'ую']),
 ('покрытосемянно', ['по', 'кры', 'т', 'о', 'сем', 'ян', 'н', 'о']),
 ('переплавившемся', ['пере', 'плав', 'и', 'вш', 'ем', 'ся']),
 ('вспомянете', ['вс', 'по', 'мя', 'н', 'ете']),
 ('оближет', ['об', 'лиж', 'ет'])]

In [12]:
## я не помню уже зачем я это делал

def find_and_replace(word:str, splits:list[str], char1="е", char2="ё"):

    assert len(word) == len("".join(splits))

    ptr = 0
    new_splits = []

    for split in splits:
        char1_count = split.count(char1)

        if char1_count != 0:
            rel_positions = [i for i, ch in enumerate(split) if ch == char1]
            abs_positions = [i + ptr for i in rel_positions]

            for abs, rel in zip(abs_positions, rel_positions):
                # print(f"{word}[{abs}] == {word[abs]}")
                if word[abs] == char2:
                    split = split[:rel] + char2 + split[rel+1:]


        ptr += len(split)
        new_splits.append(split)

    return new_splits


In [13]:
find_and_replace(
    "пёрёёвёртышё",
    ["пере", "евер", "тыше"],
)

['пёрё', 'ёвёр', 'тышё']

In [14]:
morphodict_k = defaultdict(set)

for w, split in texts_segments:
    w_from_split = "".join(split)

    if w != w_from_split:
        morphodict_k[w_from_split].add("/".join(split))

        new_split = find_and_replace(w, split.copy())
        morphodict_k[w].add("/".join(new_split))

    else:
        morphodict_k[w].add("/".join(split))

In [15]:
assert morphodict_k["повёртываете"]
assert morphodict_k["повертываете"]
assert morphodict_k["издёргавшая"]
assert morphodict_k["издергавшая"]

In [30]:
morphodict_k["повёртываете"]

{'по/вёрт/ыва/ете'}

In [16]:
len(morphodict_k)

1422074

In [ ]:
## Тут уже помню, но данные надо привести в порядок
## АХТУНГ скорее всего далее код сломан.
## нужно чтобы segments_flat были просто [морфема, морфема, морфема, ..., морфема]
segments_flat = []

for word, set_morphemes in morphodict_k.items():
    for morphemes in set_morphemes:
        segments_flat.append(morphemes.split("/")[0])

        if len(morphemes) > 1:
            for morpheme in morphemes.split("/")[1:]:
                segments_flat.append("##"+morpheme)

In [19]:
# based on https://github.com/10-OASIS-01/BPEtokenizer/blob/main/tokenizer/basic.py


def get_stats(splitted_morphemes: list[list[int]]):
    counts: dict[tuple[int, int], int] = {}
    for morpheme in splitted_morphemes:
        for pair in zip(morpheme[:-1], morpheme[1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids: list[list[int]], pair: tuple[int, int], idx: int):
    new_ids = []

    for morpheme in ids:
        new_morpheme = []
        i = 0

        while i < len(morpheme):
            if (
                i < len(morpheme) - 1
                and morpheme[i] == pair[0]
                and morpheme[i + 1] == pair[1]
            ):
                new_morpheme.append(idx)
                i += 2
            else:
                new_morpheme.append(morpheme[i])
                i += 1
        new_ids.append(new_morpheme)

    return new_ids


class MorphBpeTokenizer:

    def __init__(self):

        self.merges: dict[(int, int), int] = {}
        self.vocab = dict()
        self.char2id = dict()

    def train(self, morphemes: list[str], vocab_size:int, verbose=False):

        merges: dict[tuple[int, int], int] = {}
        initial_vocab = (
            "абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
            "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
        )
        vocab = {idx: initial_vocab[idx] for idx in range(len(initial_vocab))}
        self.char2id = {char: idx for idx, char in vocab.items()}
        idx = max(vocab) + 1
        num_merges = vocab_size - len(vocab)

        ids = [[self.char2id[char] for char in morpheme] for morpheme in morphemes]

        for i in tqdm(range(num_merges), total=num_merges):

            # count up the number of times every consecutive pair appears
            stats = get_stats(ids)
            # find the pair with the highest count
            pair = max(stats, key=stats.get)
            # mint a new token: assign it the next available id
            idx = idx + i
            # replace all occurrences of pair in ids with idx
            ids = merge(ids, pair, idx)
            # save the merge
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            # save class variables
            self.merges = merges  # used in encode()
            self.vocab = vocab  # used in decode()

    def encode(self, word: str):
        ids = [self.char2id[char] for char in word]

        while len(ids) >= 2:
            # find the pair with the lowest merge index
            stats = get_stats([ids])
            # Key point: Here, we are not selecting the byte pair based on frequency, but based on the merge index in the self.merges dictionary.
            # self.merges stores the merge priority or order for each byte pair. The min(stats, key=...) selects the byte pair with the smallest merge index,
            # which corresponds to the pair that needs to be merged first, not the pair with the lowest frequency.
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            # subtle: if there are no more merges available, the key will
            # result in an inf for every single pair, and the min will be
            # just the first pair in the list, arbitrarily
            # we can detect this terminating case by a membership check
            if pair not in self.merges:
                break  # nothing else can be merged anymore
            # otherwise let's merge the best pair (lowest merge index)
            idx = self.merges[pair]
            ids = merge([ids], pair, idx)[0]
        return ids

    def decode(self, ids: list[int]) -> list[str]:
        return [self.vocab[i] for i in ids]


In [20]:
# TODO: Алгоритм работает, но очень медленно. Нужно оптимизация
#       Варианты:
#           1) нормальный: оптимизировать мержи
#           2) ленивый: навайбкодить то же самое на C

tok = MorphBpeTokenizer()
tok.train(segments_flat[:20000], 2000)

cases = [
    "прикопанную",
    "водообеспеченность",
    "linguistics!",
    "свежевскопанный",
    "свежевскопанная",
    "свежевскопанные",
    "свежевскопанными",
]

for word in cases:
    print(word)
    token_ids = tok.encode(word)
    print(token_ids)
    print(tok.decode(token_ids))
    print()

100%|██████████| 1850/1850 [00:08<00:00, 215.26it/s]

прикопанную
[853, 38653, 0, 816, 615]
['при', 'коп', 'а', 'нн', 'ую']

водообеспеченность
[1278, 1690, 450, 377296, 287811, 745, 29070]
['во', 'до', 'об', 'ес', 'печ', 'енн', 'ость']

linguistics!
[109, 106, 111, 104, 118, 106, 116, 117, 106, 100, 116, 66]
['l', 'i', 'n', 'g', 'u', 'i', 's', 't', 'i', 'c', 's', '!']

свежевскопанный
[18, 8406, 102528, 606801, 0, 816, 3390]
['с', 'ве', 'жев', 'скоп', 'а', 'нн', 'ый']

свежевскопанная
[18, 8406, 102528, 606801, 7171, 270, 32]
['с', 'ве', 'жев', 'скоп', 'ан', 'на', 'я']

свежевскопанные
[18, 8406, 102528, 606801, 0, 816, 3720]
['с', 'ве', 'жев', 'скоп', 'а', 'нн', 'ые']

свежевскопанными
[18, 8406, 102528, 606801, 0, 816, 3978]
['с', 'ве', 'жев', 'скоп', 'а', 'нн', 'ыми']

